# Kathakaar — Cultural Heritage Style LoRA (SDXL)

Fine-tune an open image model (Stable Diffusion XL) with **LoRA** on a small set of
heritage images you curate, so it learns *your* cultural aesthetic. Then generate
place / era / theme-conditioned images you can later animate into your cinematic.

**Runs on a free GPU** (Google Colab or Kaggle, NVIDIA T4). One training run ≈ 30–60 min.

### Honest expectations
- This trains an **image** style model, not a video model. Video comes in the next step
  (image→video), and even then clips are short (2–5s) and stylized — not a finished film.
- Quality depends almost entirely on the **images you curate**. 20–40 good, consistent
  images beat 200 random ones.
- **Licensing + cultural sensitivity matter.** Use freely-licensed images (e.g. Wikimedia
  Commons public-domain / CC), keep attribution, and be respectful of sacred or sensitive
  heritage. This curation *is* the culturally-appropriate part — it's the real work, and
  it's what makes your project yours.

> First time on Colab: **Runtime → Change runtime type → T4 GPU**, then run cells top to bottom.

## 1. Check the GPU and install dependencies

In [ ]:
!nvidia-smi
!pip -q install "diffusers>=0.27" "transformers>=4.40" "accelerate>=0.30" \
    "peft>=0.11" datasets bitsandbytes safetensors pillow
print("\nDependencies installed.")

## 2. Get the official LoRA training script
We use Hugging Face `diffusers`' DreamBooth-LoRA script for SDXL (well-tested, T4-friendly).

In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py -O train_dreambooth_lora_sdxl.py
import os; print('script ready:', os.path.exists('train_dreambooth_lora_sdxl.py'))

## 3. Prepare your heritage image set
Put **20–40 images** of your subject into the `instance` folder. Two options:

**A) Upload manually (recommended for control):** use the Colab Files panel (folder icon) to
upload images into `/content/instance`.

**B) Auto-fetch a starter set from Wikimedia Commons** (run the cell below). Always review the
results and keep only on-topic, well-licensed images. Attribution is printed for each.

In [ ]:
import os, json, urllib.parse, urllib.request

INSTANCE_DIR = '/content/instance'
os.makedirs(INSTANCE_DIR, exist_ok=True)

# ---- EDIT THIS: the heritage subject you want the LoRA to learn ----
SUBJECT = 'Konark Sun Temple'
MAX_IMAGES = 25
# --------------------------------------------------------------------

UA = {'User-Agent': 'KathakaarLoRA/1.0 (educational portfolio project)'}

def commons_image_urls(term, n):
    api = ('https://commons.wikimedia.org/w/api.php?action=query&format=json'
           '&generator=search&gsrnamespace=6&gsrlimit=%d&gsrsearch=%s'
           '&prop=imageinfo&iiprop=url|extmetadata') % (n, urllib.parse.quote(term))
    req = urllib.request.Request(api, headers=UA)
    data = json.load(urllib.request.urlopen(req, timeout=20))
    out = []
    for page in data.get('query', {}).get('pages', {}).values():
        info = (page.get('imageinfo') or [{}])[0]
        url = info.get('url')
        if url and url.lower().endswith(('.jpg', '.jpeg', '.png')):
            lic = (info.get('extmetadata', {}).get('LicenseShortName', {}) or {}).get('value', 'unknown')
            out.append((url, lic))
    return out

urls = commons_image_urls(SUBJECT, MAX_IMAGES)
print('found', len(urls), 'candidates for', SUBJECT)
for i, (url, lic) in enumerate(urls):
    try:
        req = urllib.request.Request(url, headers=UA)
        ext = os.path.splitext(url)[1].split('?')[0]
        with urllib.request.urlopen(req, timeout=30) as r, open(f'{INSTANCE_DIR}/img_{i:02d}{ext}', 'wb') as f:
            f.write(r.read())
        print(f'  [{i:02d}] license={lic}')
    except Exception as e:
        print('  skip', i, e)
print('\nSaved to', INSTANCE_DIR, '— now REVIEW them and delete any off-topic images.')

## 4. Train the LoRA
Settings are tuned for a **free T4 (≈16GB)**: 768px, batch 1, 8-bit Adam, gradient checkpointing.
`khvheritage` is your unique **trigger word** — you'll put it in prompts later to invoke the style.

_If you hit out-of-memory:_ lower `--resolution` to 640, or switch to the lighter SD 1.5 script
(`train_dreambooth_lora.py` with `runwayml/stable-diffusion-v1-5`).

In [ ]:
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --pretrained_vae_model_name_or_path='madebyollin/sdxl-vae-fp16-fix' \
  --instance_data_dir='/content/instance' \
  --instance_prompt='a photo in khvheritage style' \
  --output_dir='/content/kathakaar-lora' \
  --resolution=768 --train_batch_size=1 --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 --lr_scheduler='constant' --lr_warmup_steps=0 \
  --max_train_steps=800 --rank=16 --seed=42 \
  --gradient_checkpointing --use_8bit_adam --mixed_precision='fp16' \
  --checkpointing_steps=400

## 5. Generate images with your trained LoRA
Combine the trigger word with **place + era + theme** — exactly the inputs Kathakaar already collects.

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', vae=vae, torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights('/content/kathakaar-lora')

# place + era + theme  (mirror your Kathakaar inputs)
prompt = ('khvheritage style, the Konark Sun Temple, 13th century medieval India, '
          'temple architecture and carved stone wheels, cinematic golden dawn light, '
          'epic wide shot, highly detailed')
neg = 'modern buildings, cars, text, watermark, lowres, blurry'

img = pipe(prompt, negative_prompt=neg, num_inference_steps=30, guidance_scale=7.0).images[0]
img.save('/content/sample.png')
img

## 6. Save it / what's next
- Your LoRA is a small `.safetensors` in `/content/kathakaar-lora` — download it from the Files panel.
- **Push to Hugging Face Hub** (optional) to use it from anywhere:
  ```python
  from huggingface_hub import notebook_login; notebook_login()
  pipe.save_lora_weights('/content/kathakaar-lora')  # then upload the folder to a HF repo
  ```

### The next notebook (image → video)
Feed these generated images into an open image-to-video model (e.g. **Stable Video Diffusion**,
**Wan-I2V**, or **LTX-Video**) to get short moving clips, then stitch clips + your narration audio
with `ffmpeg`. That output plugs into Kathakaar's existing `/api/render` hook.

### Resume line you're earning here
> *Fine-tuned Stable Diffusion XL with LoRA on a curated cultural-heritage dataset; built a
> place/era/theme-conditioned generation pipeline feeding an image-to-video cinematic.*

That's a real generative-ML skill — training and controlling a model, not just calling an API.